# 🧠 SQL Query Expert v3 — Natural Language → PostgreSQL (Vertex AI)

Ask a question in plain English → get a **safe, validated PostgreSQL query** + its results.

**New in v3:**
- 🔀 **Ambiguity detection** — if your question could be answered from MORE THAN ONE table
  (e.g. a bid amount stored in two places), the system now **tells you the options and asks
  which one you meant** — or runs **both** and shows both results — instead of silently
  picking one.

**Strict checking (from v2):**
- 🚧 **Answerability gate** — BEFORE any SQL is written, a separate check decides whether your
  question can even be answered from your database. Greetings, general knowledge, or questions
  about tables/entities you don't have are **refused with a reason**, not answered with made-up SQL.
- 🧾 **Column-level validation** — every `alias.column` in the generated SQL is checked against
  the real columns of the real table. Invented columns are rejected.
- 🗄️ **Real-table requirement** — a query that doesn't touch at least one actual table in your
  database is rejected (no more `SELECT 2+2` slipping through).
- 🧪 **Guardrail self-test cell** — run it and *watch* bad inputs get rejected, so you can trust
  the checks.

```
question → answerability gate → ambiguity check (ask you / run both)
        → Gemini → keyword/table/column checks
        → EXPLAIN dry-run → read-only execution → output checks
```

**How to use:** fill in Step 2, then run every cell top-to-bottom, in order.
⚠️ If you skip a step, later steps will fail — each step builds on the previous one.

---
## Step 1 — Install the libraries

In [ ]:
%pip install -q google-genai psycopg2-binary pandas

print("✅ Libraries installed. If pip printed a 'restart kernel' notice, restart the kernel now.")

---
## Step 2 — Configuration

Edit for **your** GCP project and **your** PostgreSQL database.

**Authentication for Vertex AI** (do ONE of these first):
- Your own machine: run `gcloud auth application-default login` in a terminal.
- Google Colab: run `from google.colab import auth; auth.authenticate_user()` in a cell.
- Server: set `GOOGLE_APPLICATION_CREDENTIALS=/path/to/service-account.json`.

Make sure the **Vertex AI API is enabled** in the GCP console for your project.

In [ ]:
# ---------- Google Cloud / Vertex AI ----------
PROJECT_ID = "your-gcp-project-id"     # ⬅️ CHANGE
LOCATION   = "us-central1"
MODEL_ID   = "gemini-2.5-flash"        # try "gemini-2.5-pro" for harder questions

# ---------- PostgreSQL ----------
DB_CONFIG = {
    "host":     "localhost",           # ⬅️ CHANGE if needed
    "port":     5432,
    "dbname":   "your_database",       # ⬅️ CHANGE
    "user":     "postgres",            # ⬅️ CHANGE
    "password": "your_password",       # ⬅️ CHANGE
}
DB_SCHEMA = "public"
MAX_ROWS  = 500

print("✅ Configuration loaded (checked for real in the next steps).")

---
## Step 3 — Connect to PostgreSQL & health checks

Verifies: **(1)** the connection works, **(2)** tables exist, **(3)** each table has data
(empty tables are flagged so a correct-but-empty result never surprises you).

In [ ]:
import pandas as pd
import psycopg2
import psycopg2.extras
from psycopg2 import sql as pgsql

def get_conn():
    return psycopg2.connect(**DB_CONFIG)

try:
    with get_conn() as conn, conn.cursor() as cur:
        cur.execute("SELECT current_database(), version();")
        dbname, version = cur.fetchone()
    print(f"✅ Connected to database '{dbname}'")
    print(f"   Server: {version.split(',')[0]}")
except Exception as e:
    raise SystemExit(f"❌ Could not connect to PostgreSQL: {e}\n"
                     "   → Check host/port/dbname/user/password in Step 2.")

In [ ]:
def health_check():
    with get_conn() as conn, conn.cursor() as cur:
        cur.execute(
            """SELECT table_name FROM information_schema.tables
               WHERE table_schema = %s AND table_type = 'BASE TABLE'
               ORDER BY table_name""",
            (DB_SCHEMA,),
        )
        tables = [r[0] for r in cur.fetchall()]
        if not tables:
            print(f"⚠️  No tables found in schema '{DB_SCHEMA}'.")
            print("   → Run the OPTIONAL sample-data cell below, or point Step 2 at a real database.")
            return []
        report = []
        for t in tables:
            cur.execute(pgsql.SQL("SELECT COUNT(*) FROM {}.{}").format(
                pgsql.Identifier(DB_SCHEMA), pgsql.Identifier(t)))
            n = cur.fetchone()[0]
            report.append({"table": t, "rows": n,
                           "status": "✅ has data" if n > 0 else "⚠️ EMPTY"})
    display(pd.DataFrame(report))
    empty = [r["table"] for r in report if r["rows"] == 0]
    if empty:
        print(f"⚠️  Empty tables (queries on them WILL return 0 rows): {', '.join(empty)}")
    else:
        print("✅ All tables contain data.")
    return tables

KNOWN_TABLES = health_check()

### (Optional) Create a small demo database

Only if the health check found **no tables**. Off by default so it can never touch real data.

In [ ]:
CREATE_SAMPLE_DATA = False   # ⬅️ set True only for a demo playground

if CREATE_SAMPLE_DATA:
    ddl = """
    CREATE TABLE IF NOT EXISTS customers (
        id SERIAL PRIMARY KEY, name TEXT NOT NULL, email TEXT UNIQUE,
        city TEXT, created_at TIMESTAMP DEFAULT NOW());
    CREATE TABLE IF NOT EXISTS products (
        id SERIAL PRIMARY KEY, name TEXT NOT NULL, category TEXT,
        price NUMERIC(10,2) NOT NULL);
    CREATE TABLE IF NOT EXISTS orders (
        id SERIAL PRIMARY KEY, customer_id INT REFERENCES customers(id),
        created_at TIMESTAMP DEFAULT NOW(), status TEXT DEFAULT 'completed');
    CREATE TABLE IF NOT EXISTS order_items (
        id SERIAL PRIMARY KEY, order_id INT REFERENCES orders(id),
        product_id INT REFERENCES products(id), quantity INT NOT NULL,
        unit_price NUMERIC(10,2) NOT NULL);
    INSERT INTO customers (name, email, city) VALUES
      ('Asha Rao','asha@example.com','Bengaluru'),
      ('Vikram Singh','vikram@example.com','Delhi'),
      ('Meera Iyer','meera@example.com','Chennai')
    ON CONFLICT (email) DO NOTHING;
    INSERT INTO products (name, category, price)
      SELECT * FROM (VALUES
        ('Laptop','electronics',55000.00),('Mouse','electronics',799.00),
        ('Desk Chair','furniture',6500.00),('Notebook','stationery',120.00)) v
      WHERE NOT EXISTS (SELECT 1 FROM products);
    INSERT INTO orders (customer_id, status)
      SELECT * FROM (VALUES (1,'completed'),(1,'completed'),(2,'completed'),(3,'pending')) v
      WHERE NOT EXISTS (SELECT 1 FROM orders);
    INSERT INTO order_items (order_id, product_id, quantity, unit_price)
      SELECT * FROM (VALUES (1,1,1,55000.00),(1,2,2,799.00),(2,3,1,6500.00),
                            (3,4,10,120.00),(4,2,1,799.00)) v
      WHERE NOT EXISTS (SELECT 1 FROM order_items);
    """
    with get_conn() as conn, conn.cursor() as cur:
        cur.execute(ddl)
        conn.commit()
    print("✅ Sample data created. Re-running health check...")
    KNOWN_TABLES = health_check()
else:
    print("Skipped (CREATE_SAMPLE_DATA is False).")

---
## Step 4 — Extract the schema (tables + columns catalog)

Builds two things: **`SCHEMA_TEXT`** (human-readable schema that goes into every prompt) and
**`TABLE_COLUMNS`** (a `{table: {columns}}` catalog the validator uses to reject invented
tables/columns).

In [ ]:
def load_schema():
    lines = []
    table_columns = {}
    with get_conn() as conn, conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        for t in KNOWN_TABLES:
            cur.execute(
                """SELECT column_name, data_type, is_nullable
                   FROM information_schema.columns
                   WHERE table_schema=%s AND table_name=%s ORDER BY ordinal_position""",
                (DB_SCHEMA, t))
            cols = cur.fetchall()
            table_columns[t.lower()] = {c["column_name"].lower() for c in cols}

            cur.execute(
                """SELECT a.attname AS col FROM pg_index i
                   JOIN pg_attribute a ON a.attrelid=i.indrelid AND a.attnum=ANY(i.indkey)
                   WHERE i.indrelid=%s::regclass AND i.indisprimary""",
                (f'{DB_SCHEMA}."{t}"',))
            pks = {r["col"] for r in cur.fetchall()}

            cur.execute(
                """SELECT kcu.column_name AS col, ccu.table_name AS ref_table,
                          ccu.column_name AS ref_col
                   FROM information_schema.table_constraints tc
                   JOIN information_schema.key_column_usage kcu
                     ON tc.constraint_name=kcu.constraint_name AND tc.table_schema=kcu.table_schema
                   JOIN information_schema.constraint_column_usage ccu
                     ON ccu.constraint_name=tc.constraint_name AND ccu.table_schema=tc.table_schema
                   WHERE tc.constraint_type='FOREIGN KEY'
                     AND tc.table_schema=%s AND tc.table_name=%s""",
                (DB_SCHEMA, t))
            fks = cur.fetchall()

            lines.append(f"TABLE {t} (")
            for c in cols:
                pk = " PRIMARY KEY" if c["column_name"] in pks else ""
                nn = "" if c["is_nullable"] == "YES" else " NOT NULL"
                lines.append(f"    {c['column_name']} {c['data_type']}{nn}{pk}")
            for fk in fks:
                lines.append(f"    FOREIGN KEY ({fk['col']}) REFERENCES {fk['ref_table']}({fk['ref_col']})")
            lines.append(")\n")
    return "\n".join(lines), table_columns

SCHEMA_TEXT, TABLE_COLUMNS = load_schema() if KNOWN_TABLES else ("", {})
print(SCHEMA_TEXT if SCHEMA_TEXT else "⚠️ Schema is empty — fix Step 3 first.")

---
## Step 5 — Connect to Vertex AI (Gemini)

In [ ]:
from google import genai

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

try:
    test = client.models.generate_content(
        model=MODEL_ID, contents="Reply with exactly: Vertex AI is connected!")
    print("✅", test.text.strip())
except Exception as e:
    raise SystemExit(f"❌ Vertex AI call failed: {e}\n"
                     "   → Check PROJECT_ID / LOCATION, authentication (Step 2 notes),\n"
                     "     and that the Vertex AI API is enabled.")

---
## Step 6 — 🚧 The answerability gate + 🔀 ambiguity check

Two model-powered checks that run BEFORE any SQL is generated:

1. **Answerability gate** — refuses questions your database can't answer (wrong topic,
   missing table, chit-chat) with the reason and what's missing.
2. **Ambiguity check** — if the SAME information could come from more than one table or
   column (e.g. `bids.amount` vs `auctions.total_bid_amount`), it lists the interpretations
   and lets you choose — or answers from **both** — instead of silently picking one.

In [ ]:
import json as _json
import re

def _parse_json_reply(text):
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```[a-zA-Z]*\s*|\s*```$", "", text).strip()
    try:
        return _json.loads(text)
    except _json.JSONDecodeError:
        s, e = text.find("{"), text.rfind("}")
        if s != -1 and e != -1:
            return _json.loads(text[s:e+1])
        raise

def is_answerable(question):
    """Strict gatekeeper: can this question be answered from THIS schema only?"""
    prompt = f"""You are a strict gatekeeper for a SQL system. Below is the COMPLETE database schema.

{SCHEMA_TEXT}

Question: {question}

Decide if this question can be answered using ONLY the tables and columns above.
Rules:
- Greetings, chit-chat, math, coding help, or general knowledge → NOT answerable.
- Mentions data/entities/tables/columns that do not exist in this schema → NOT answerable,
  and list the missing things.
- Only mark answerable if a correct SELECT query over these exact tables can answer it.
Respond with RAW JSON only:
{{"answerable": true or false, "reason": "<one sentence>", "missing": ["<things not in the schema, if any>"]}}"""
    resp = client.models.generate_content(model=MODEL_ID, contents=prompt)
    try:
        data = _parse_json_reply(resp.text)
    except Exception:
        # If the gate itself misbehaves, fail CLOSED (refuse), never open.
        return {"answerable": False, "reason": "Gate response was unreadable — refusing to guess.", "missing": []}
    return {"answerable": bool(data.get("answerable")),
            "reason": data.get("reason", ""),
            "missing": data.get("missing", []) or []}

def check_ambiguity(question):
    """Could this question be answered from MORE THAN ONE table/column?"""
    prompt = f"""You are helping disambiguate a database question. Complete schema:

{SCHEMA_TEXT}

Question: {question}

Could this question reasonably be answered in MORE THAN ONE way from this schema — for
example, the same information (an amount, a total, a date, a name) exists in two or more
different tables or columns? Only report REAL alternatives that exist in this schema.

Respond with RAW JSON only:
{{"ambiguous": true or false,
  "interpretations": [
    {{"summary": "<short plain-English description of this interpretation>",
      "tables": ["<table(s) it would use>"],
      "clarified_question": "<the question rewritten so it is 100% unambiguous>"}}
  ]}}
If NOT ambiguous, return "ambiguous": false with exactly ONE interpretation (the obvious one)."""
    resp = client.models.generate_content(model=MODEL_ID, contents=prompt)
    try:
        data = _parse_json_reply(resp.text)
    except Exception:
        return {"ambiguous": False, "interpretations": []}
    interps = [i for i in (data.get("interpretations") or []) if isinstance(i, dict)]
    return {"ambiguous": bool(data.get("ambiguous")) and len(interps) > 1,
            "interpretations": interps}

print("✅ Answerability gate + ambiguity check ready.")

---
## Step 7 — The SQL-generation prompt

The prompt now also embeds the **exact list of allowed table names** and forbids guessing.
👉 Once things work, add few-shot examples about *your* tables — biggest accuracy win.

In [ ]:
SYSTEM_PROMPT = """You are an expert PostgreSQL query writer. Convert the user's
natural-language question into ONE correct, read-only PostgreSQL query.

Rules (follow ALL of them):
1. PostgreSQL dialect only.
2. Output ONLY SELECT statements (WITH ... SELECT allowed). NEVER INSERT, UPDATE, DELETE,
   DROP, ALTER, TRUNCATE, CREATE, GRANT or anything that changes data or schema.
3. Use ONLY tables and columns that appear in the schema below. NEVER invent or guess names.
   If the question mentions anything not in the schema, return an empty "sql" and explain
   what is missing. Do NOT substitute a similar-looking table instead.
4. Use explicit JOINs and table aliases; qualify columns when joining.
5. Respond with RAW JSON only (no markdown fences), exactly:
   {"sql": "<query or empty string>", "explanation": "<one short paragraph>"}
"""

FEW_SHOT_EXAMPLES = """
Question: How many orders did each customer place?
{"sql": "SELECT c.id, c.name, COUNT(o.id) AS order_count FROM customers c LEFT JOIN orders o ON o.customer_id = c.id GROUP BY c.id, c.name ORDER BY order_count DESC;", "explanation": "Counts orders per customer, including customers with zero orders."}

Question: Which products have never been ordered?
{"sql": "SELECT p.id, p.name FROM products p WHERE NOT EXISTS (SELECT 1 FROM order_items oi WHERE oi.product_id = p.id);", "explanation": "Anti-join keeps products with no matching order_items row."}

Question: Show me the employee salaries
{"sql": "", "explanation": "The schema has no employees or salaries table, so this cannot be answered."}
"""

def build_prompt(question, previous_error=None):
    allowed = ", ".join(sorted(KNOWN_TABLES))
    p = (SYSTEM_PROMPT
         + f"\nALLOWED TABLES (use no others): {allowed}\n"
         + "\nDatabase schema:\n" + SCHEMA_TEXT
         + "\nExamples:\n" + FEW_SHOT_EXAMPLES
         + f"\nQuestion: {question}\nAnswer with raw JSON only.")
    if previous_error:
        p += (f"\n\nIMPORTANT: your previous attempt failed with this error:\n{previous_error}"
              "\nFix the query and answer with raw JSON only.")
    return p

print("✅ Prompt builder ready.")

---
## Step 8 — Hardened validators

Five layers between the model and your data:

| Layer | Catches |
|---|---|
| **1. Read-only / single statement** | `DELETE`, `DROP`, hidden `;DROP`, any write |
| **2. Table existence** | tables that don't exist in your DB (CTE-aware) |
| **3. Column existence** | `alias.column` pairs where the column isn't in that table |
| **4. `EXPLAIN` dry-run** | everything else — PostgreSQL plans the query without running it |
| **5. Read-only execution** | run-time safety: `readonly` session, always rolled back |

A query must also reference **at least one real table** in your database, otherwise it is rejected.

In [ ]:
FORBIDDEN = {"INSERT","UPDATE","DELETE","DROP","ALTER","TRUNCATE","CREATE",
             "GRANT","REVOKE","MERGE","CALL","COPY","VACUUM","REINDEX","COMMENT","SET"}

_SQL_STOPWORDS = {"ON","WHERE","GROUP","ORDER","LEFT","RIGHT","INNER","OUTER","JOIN",
                  "CROSS","FULL","LIMIT","HAVING","UNION","USING","AS","SELECT","WITH",
                  "NATURAL","LATERAL","OFFSET","FETCH"}

def _strip_literals(sql_text):
    s = re.sub(r"/\*.*?\*/", " ", sql_text, flags=re.DOTALL)
    s = re.sub(r"--[^\n]*", " ", s)
    s = re.sub(r"'(?:[^']|'')*'", "''", s)
    # EXTRACT(month FROM x), SUBSTRING(x FROM 1), TRIM(... FROM x) contain a fake FROM —
    # blank their argument lists so the table scanner doesn't trip on them
    s = re.sub(r"\b(EXTRACT|SUBSTRING|TRIM|OVERLAY|POSITION)\s*\([^()]*\)", r"\1()", s, flags=re.I)
    return s

def _scan_tables(cleaned):
    """Return (used_tables, alias_map, cte_names) from FROM/JOIN clauses."""
    cte_names = {m.group(1).lower() for m in
                 re.finditer(r"\b([A-Za-z_]\w*)\s+AS\s*\(", cleaned, re.I)}
    used, alias_map = set(), {}
    for m in re.finditer(
            r"\b(?:FROM|JOIN)\s+([A-Za-z_][\w.]*)(?:\s+(?:AS\s+)?([A-Za-z_]\w*))?",
            cleaned, re.I):
        tbl = m.group(1).split(".")[-1].strip('"').lower()
        used.add(tbl)
        alias = m.group(2)
        if alias and alias.upper() not in _SQL_STOPWORDS:
            alias_map[alias.lower()] = tbl
        alias_map.setdefault(tbl, tbl)   # a table is its own alias
    return used, alias_map, cte_names

def validate_sql(sql_text):
    """Layers 1–3. Returns (ok, reason)."""
    if not sql_text or not sql_text.strip():
        return False, "Empty query."
    cleaned = _strip_literals(sql_text).strip()

    if ";" in cleaned.rstrip(";"):
        return False, "Multiple SQL statements are not allowed."
    first = re.match(r"\s*([A-Za-z]+)", cleaned)
    first_kw = first.group(1).upper() if first else ""
    if first_kw not in ("SELECT", "WITH"):
        return False, f"Only SELECT/WITH queries are allowed (got '{first_kw}')."
    for kw in FORBIDDEN:
        if re.search(rf"\b{kw}\b", cleaned.upper()):
            return False, f"Forbidden keyword found: {kw}."

    known = {t.lower() for t in KNOWN_TABLES}
    used, alias_map, cte_names = _scan_tables(cleaned)

    unknown = used - known - cte_names
    if unknown:
        return False, (f"Query references tables that do NOT exist in your database: "
                       f"{', '.join(sorted(unknown))}. Your tables: {', '.join(sorted(known))}.")
    if not (used & known):
        return False, ("Query does not use any table from your database — refusing "
                       "(questions must be about YOUR data).")

    # Layer 3: every alias.column must exist in that table
    for m in re.finditer(r"\b([A-Za-z_]\w*)\.([A-Za-z_]\w*)\b", cleaned):
        alias, col = m.group(1).lower(), m.group(2).lower()
        tbl = alias_map.get(alias)
        if tbl in TABLE_COLUMNS and col != "*" and col not in TABLE_COLUMNS[tbl]:
            return False, (f"Column '{col}' does not exist in table '{tbl}'. "
                           f"Available columns: {', '.join(sorted(TABLE_COLUMNS[tbl]))}.")
    return True, ""

def tables_in(sql_text):
    used, _, cte = _scan_tables(_strip_literals(sql_text))
    return sorted(used - cte)

def explain_check(sql_text):
    """Layer 4: PostgreSQL plans the query WITHOUT executing it."""
    try:
        with get_conn() as conn:
            conn.set_session(readonly=True)
            with conn.cursor() as cur:
                cur.execute("EXPLAIN " + sql_text)
            conn.rollback()
        return True, ""
    except Exception as e:
        return False, str(e).strip()

def add_limit(sql_text, max_rows=MAX_ROWS):
    if re.search(r"\bLIMIT\s+\d+", sql_text, re.I):
        return sql_text
    return sql_text.rstrip().rstrip(";") + f" LIMIT {max_rows};"

print("✅ Hardened validators ready.")

---
## Step 9 — 🧪 Self-test the guardrails (watch bad input get rejected)

Run this cell and confirm every line says **REJECTED**. If your database has different table
names, that's fine — the point is that none of these dangerous or invalid queries pass.

In [ ]:
_bad = [
    ("DELETE FROM " + (KNOWN_TABLES[0] if KNOWN_TABLES else "x"), "a write statement"),
    ("SELECT 1; DROP TABLE users",                                "hidden second statement"),
    ("SELECT * FROM table_that_does_not_exist_xyz",               "nonexistent table"),
    ("SELECT 2+2",                                                "no table from your DB"),
    ("UPDATE t SET a=1",                                          "an UPDATE"),
]
if KNOWN_TABLES:
    t0 = KNOWN_TABLES[0]
    _bad.append((f"SELECT z.made_up_column_xyz FROM {t0} z", "invented column on a real table"))

print("Guardrail self-test:")
all_ok = True
for q, why in _bad:
    ok, reason = validate_sql(q)
    verdict = "❌ PASSED (BUG!)" if ok else "✅ REJECTED"
    if ok:
        all_ok = False
    print(f"  {verdict}  [{why}]  →  {reason if reason else q}")
print("\n" + ("✅ All guardrails working." if all_ok else "❌ Some guardrail failed — do NOT use, re-check steps."))

---
## Step 10 — The full pipeline: `ask("your question")`

Order of operations for every question:

1. **Gate** — refuses questions your database can't answer (with the reason).
2. **Ambiguity check** — if the answer could come from more than one table, you choose
   (or it answers from both).
3. **Generate** — Gemini writes the SQL.
4. **Validate** — read-only, tables exist, columns exist.
5. **EXPLAIN** — PostgreSQL confirms the query is plannable, without running it.
6. **Execute** — read-only, rolled back, row-capped.
7. **Check output** — warns on empty results, all-NULL columns, capped rows.

**Choosing how ambiguity is handled** with the `on_ambiguity` option:

| `ask("...", on_ambiguity=...)` | Behaviour |
|---|---|
| `"ask"` (default) | lists the interpretations and asks you to type a number |
| `"both"` | runs EVERY interpretation and shows each result, labelled |
| `None` | old behaviour: let the model pick (not recommended) |

If a step fails, the error goes back to Gemini for another attempt (up to 3).

In [ ]:
def generate_sql(question, previous_error=None):
    resp = client.models.generate_content(
        model=MODEL_ID, contents=build_prompt(question, previous_error))
    try:
        data = _parse_json_reply(resp.text)
    except Exception:
        return "", "Model returned unreadable output."
    return (data.get("sql") or "").strip(), (data.get("explanation") or "").strip()

def run_query(sql_text):
    with get_conn() as conn:
        conn.set_session(readonly=True)
        with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            cur.execute(add_limit(sql_text))
            rows = cur.fetchall()
        conn.rollback()
    return pd.DataFrame(rows)

def validate_output(df):
    if df.empty:
        print("⚠️  Query ran fine but returned 0 rows — either the filter matched nothing,")
        print("    or the table is empty (see Step 3's health report).")
        return
    null_cols = [c for c in df.columns if df[c].isna().all()]
    if null_cols:
        print(f"⚠️  Columns entirely NULL: {', '.join(null_cols)} (often an unmatched LEFT JOIN).")
    if len(df) >= MAX_ROWS:
        print(f"⚠️  Result capped at {MAX_ROWS} rows (raise MAX_ROWS in Step 2 if needed).")
    print(f"✅ {len(df)} row(s), {len(df.columns)} column(s).")

def _answer(question, execute=True, max_attempts=3):
    """Generate → validate → EXPLAIN → execute, with self-correction retries."""
    error = None
    for attempt in range(1, max_attempts + 1):
        sql_text, explanation = generate_sql(question, error)
        if not sql_text:
            print(f"🤷 The model could not answer from your schema.\n   Reason: {explanation}")
            return None

        ok, reason = validate_sql(sql_text)
        if not ok:
            print(f"   attempt {attempt}: rejected → {reason}")
            error = reason
            continue

        ok, reason = explain_check(sql_text)
        if not ok:
            print(f"   attempt {attempt}: PostgreSQL rejected the plan → {reason}")
            error = reason
            continue

        print("─" * 60)
        print("📝 SQL:\n" + sql_text)
        print("\n📋 Tables used: " + ", ".join(f"{t} ✅" for t in tables_in(sql_text)))
        print("💡 " + explanation)
        print("─" * 60)
        if not execute:
            return sql_text

        try:
            df = run_query(sql_text)
        except Exception as e:
            error = str(e).strip()
            print(f"   attempt {attempt}: execution failed → {error}")
            continue

        validate_output(df)
        return df

    print(f"❌ Gave up after {max_attempts} attempts. Last error: {error}")
    return None

def ask(question, execute=True, max_attempts=3, on_ambiguity="ask"):
    if not KNOWN_TABLES:
        print("❌ No tables available — complete Step 3 first.")
        return None

    # 1) the gate — refuse before generating anything
    gate = is_answerable(question)
    if not gate["answerable"]:
        print("🚫 REFUSED — this question cannot be answered from your database.")
        print(f"   Reason: {gate['reason']}")
        if gate["missing"]:
            print(f"   Not in your schema: {', '.join(str(x) for x in gate['missing'])}")
        print(f"   Tables you DO have: {', '.join(sorted(KNOWN_TABLES))}")
        return None

    # 2) ambiguity — same info in more than one table? Don't silently pick.
    if on_ambiguity in ("ask", "both"):
        amb = check_ambiguity(question)
        if amb["ambiguous"]:
            interps = amb["interpretations"]
            print(f"🔀 Your question could mean {len(interps)} different things:")
            for i, it in enumerate(interps, 1):
                tbls = ", ".join(it.get("tables", [])) or "?"
                print(f"   {i}. {it.get('summary', '(no description)')}   [tables: {tbls}]")

            if on_ambiguity == "both":
                results = {}
                for i, it in enumerate(interps, 1):
                    print("\n" + "=" * 60)
                    print(f"▶ Interpretation {i}: {it.get('summary', '')}")
                    print("=" * 60)
                    q = it.get("clarified_question") or question
                    results[it.get("summary", f"interpretation {i}")] = _answer(q, execute, max_attempts)
                return results

            # default "ask": let the user choose
            choice = input(f"   Which one did you mean? [1-{len(interps)}, Enter = 1]: ").strip()
            idx = int(choice) - 1 if choice.isdigit() and 1 <= int(choice) <= len(interps) else 0
            chosen = interps[idx]
            print(f"   → Using interpretation {idx + 1}: {chosen.get('summary', '')}")
            question = chosen.get("clarified_question") or question

    return _answer(question, execute, max_attempts)

print("✅ Pipeline ready — use ask(\"your question\") below.")

---
## Step 11 — Try it 🎉

First cell should **work**; second and third should be **refused** — that's the gate doing
its job. Then ask real questions about your data.

In [ ]:
# A real question about your data (edit for your tables):
df = ask("Show the 5 most recent orders with the customer's name")
df

In [ ]:
# These SHOULD be refused — proof it no longer answers "whatever you give it":
ask("What is the capital of France?")

In [ ]:
# Asking about a table you don't have SHOULD also be refused with what's missing:
ask("Show me all employee salaries from the payroll table")

In [ ]:
# If the same info lives in TWO tables, the default mode lists the options and
# asks you to pick a number. Or answer from BOTH tables at once:
results = ask("What is the total bids amount?", on_ambiguity="both")
# results is then a dict: {interpretation description: DataFrame}

In [ ]:
# Only generate + validate the SQL, don't run it:
sql_only = ask("Which customer spent the most in total?", execute=False)

---
## Troubleshooting

| Symptom | Fix |
|---|---|
| `could not connect to server` | DB host/port wrong or PostgreSQL not running (Step 2/3) |
| `403 PERMISSION_DENIED` from Vertex | authenticate (`gcloud auth application-default login`) + enable Vertex AI API |
| `404 model not found` | change `MODEL_ID` or region |
| Gate refuses a valid question | rephrase using your real table/column names, or relax the gate prompt in Step 6 |
| Gate allows an invalid question | the layer-2/3 validators and EXPLAIN still block bad SQL — but tell the gate prompt what to refuse |
| Correct SQL, 0 rows | table empty or filter matched nothing — check Step 3's report |

**Next level-ups:** add few-shot examples from your real schema (Step 7) · log every
(question, SQL, outcome) and reuse the good ones · for big DBs send only relevant tables per
question · wrap `ask()` in Streamlit/FastAPI when you outgrow the notebook.